## एम्बेडिङहरू

हाम्रो अघिल्लो उदाहरणमा, हामीले उच्च-आयामको ब्याग-अफ-वर्ड्स भेक्टरहरू जुन लम्बाइ `vocab_size` थियो, मा काम गर्यौं, र हामी स्पष्ट रूपमा कम-आयामिक अवस्थिति प्रतिनिधित्व भेक्टरहरूलाई विरल एक-हट प्रतिनिधित्वमा रूपान्तरण गर्दै थियौं। यो एक-हट प्रतिनिधित्व मेमोरी-कुशल छैन, साथै प्रत्येक शब्दलाई अरूबाट स्वतन्त्र रूपमा व्यवहार गरिन्छ, अर्थात् एक-हट इन्कोड गरिएका भेक्टरहरूले शब्दहरूबीच कुनै पनि अर्थगत समानता व्यक्त गर्दैनन्।

यस युनिटमा, हामी **News AG** डेटासेट अन्वेषण जारी राख्नेछौं। सुरु गर्नका लागि, डाटा लोड गरौं र अघिल्लो नोटबुकबाट केही परिभाषाहरू प्राप्त गरौं।


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## एम्बेडिङ के हो?

**एम्बेडिङ**को विचार शब्दहरूलाई कम आयाम भएका घना भेक्टरहरू द्वारा प्रतिनिधित्व गर्ने हो, जुन कुनै न कुनै रूपमा शब्दको अर्थलाई प्रतिबिम्बित गर्छ। हामी पछि कसरी अर्थपूर्ण शब्द एम्बेडिङहरू बनाउन सकिन्छ भनेर छलफल गर्नेछौं, तर अहिलेलाई एम्बेडिङहरूलाई शब्द भेक्टरको आयाम कम गर्ने तरिका भनेर मात्रै सोचौं। 

त्यसैले, एम्बेडिङ लेयरले शब्दलाई इनपुटको रूपमा लिनेछ, र निर्दिष्ट `embedding_size` को आउटपुट भेक्टर उत्पादन गर्नेछ। एक अर्थमा, यो `Linear` लेयरसँग धेरै मिल्दोजुल्दो छ, तर एक-हट इन्कोड गरिएको भेक्टर लिनुको सट्टा, यो शब्द संख्या इनपुटको रूपमा लिन सक्षम हुनेछ।

हाम्रो नेटवर्कमा पहिलो लेयरको रूपमा एम्बेडिङ लेयर प्रयोग गरेर, हामी ब्याग-ऑफ-वर्डबाट **एम्बेडिङ ब्याग** मोडेलमा सर्न सक्छौं, जसमा हामी पहिले हाम्रा पाठका प्रत्येक शब्दलाई सम्बन्धित एम्बेडिङमा रूपान्तरण गर्छौं, र त्यसपछि ती सबै एम्बेडिङहरूमा कुनै समष्टि फलन जस्तै `sum`, `average` वा `max` लागू गर्छौं।  

![पाँच शब्दहरूको अनुक्रमका लागि एम्बेडिङ वर्गीकरणकर्तालाई देखाउने छवि।](../../../../../translated_images/ne/embedding-classifier-example.b77f021a7ee67eee.webp)

हाम्रो वर्गीकरणकर्ता न्यूरल नेटवर्कले एम्बेडिङ लेयरबाट सुरु गर्नेछ, त्यसपछि समष्टि लेयर, र त्यसको माथि रेखीय वर्गीकरणकर्ता:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### चर अनुक्रम आकारसँग व्यवहार गर्दै

यस वास्तुकलाको परिणामस्वरूप, हाम्रो नेटवर्कमा मिनिब्याचहरू निश्चित तरिकाले सिर्जना गर्न आवश्यक हुनेछ। अघिल्लो इकाइमा, जब बो-ओफ-वार्ड्स प्रयोग गरिएको थियो, मिनिब्याचमा सबै BoW टेन्सरहरू बराबर आकारका थिए `vocab_size`, हाम्रो पाठ अनुक्रमको वास्तविक लम्बाईको बेवास्ता गर्दै। जब हामी शब्द एम्बेडिङ्समा जान्छौं, हामीसँग प्रत्येक पाठ नमूनामा चर संख्या शब्दहरू हुनेछ, र ती नमूनाहरूलाई मिनिब्याचहरूमा संयोजन गर्दा हामीले केही प्याडिङ लागू गर्नु पर्ने हुन्छ।

यो डाटासोर्सलाई `collate_fn` फंक्शन प्रदान गर्ने उही प्रविधि प्रयोग गरेर गर्न सकिन्छ:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### एम्बेडिङ वर्गीकर्ता तालिम

अब जब हामीले उपयुक्त डाटालोडर परिभाषित गरिसकेका छौं, हामीले अघिल्लो एकाईमा परिभाषित तालिम कार्य प्रयोग गरेर मोडेल तालिम गर्न सक्छौं:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **सूचना**: हामी यहाँ केवल २५ हजार रेकर्डहरूका लागि मात्र प्रशिक्षण गर्दैछौँ (एक पूर्ण ईपक भन्दा कम) समयको लागि, तर तपाईं प्रशिक्षण जारी राख्न सक्नुहुन्छ, धेरै ईपकहरूका लागि प्रशिक्षण गर्नको लागि एक फंक्शन लेख्न सक्नुहुन्छ, र उच्च सटीकता प्राप्त गर्न सिकाइ दर प्यारामीटरसँग प्रयोग गर्न सक्नुहुन्छ। तपाईं लगभग ९०% सटीकता प्राप्त गर्न सक्षम हुनुहुनेछ।  


### एम्बेडिङबैग लेयर र भेरियेबल-लन्थ सिक्वेन्स प्रतिनिधित्व

अघिल्लो आर्किटेक्चरमा, हामी सबै सिक्वेन्सहरूलाई समान लम्बाइमा प्याड गर्नुपर्ने थियो ताकि तिनीहरूलाई मिनिब्याचमा फिट गर्न सकियोस्। यो भेरियेबल लम्बाइका सिक्वेन्सहरू प्रतिनिधित्व गर्ने सबैभन्दा प्रभावकारी तरिका होइन - अर्को तरिका **अफसेट** भेक्टर प्रयोग गर्नु हो, जसले सबै सिक्वेन्सहरूको अफसेटहरू एउटै ठूलो भेक्टरमा राख्छ।

![अफसेट सिक्वेन्स प्रतिनिधित्व देखाउँदै गरेको छवि](../../../../../translated_images/ne/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **टिप्पणी**: माथिको चित्रमा, हामीले अक्षरहरूको सिक्वेन्स देखाएका छौं, तर हाम्रो उदाहरणमा हामी शब्दहरूको सिक्वेन्ससंग काम गर्दैछौं। तर, सिक्वेन्सलाई अफसेट भेक्टरसंग प्रतिनिधित्व गर्ने सामान्य सिद्धान्त उस्तै रहन्छ।

अफसेट प्रतिनिधित्वसँग काम गर्न हामी [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) लेयर प्रयोग गर्छौं। यो `Embedding` जस्तै छ, तर यसको इनपुटमा सामग्री भेक्टर र अफसेट भेक्टर दुबै लिन्छ, र यसमा औसत गर्ने लेयर पनि समावेश हुन्छ, जुन `mean`, `sum` वा `max` हुन सक्छ।

यहाँ परिमार्जित नेटवर्क छ जसले `EmbeddingBag` प्रयोग गर्छ:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

तालिमका लागि डेटासेट तयार गर्न, हामीले अफसेट भेक्टर तयार गर्ने रूपान्तरण प्रकार्य प्रदान गर्न आवश्यक छ:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

नोट गर्नुहोस्, अघिल्लो सबै उदाहरणहरू भन्दा फरक, हाम्रो नेटवर्क अब दुई प्यारामीटरहरू स्वीकार्छ: डेटा भेक्टर र अफसेट भेक्टर, जुन भिन्न आकारका छन्। त्यसै गरी, हाम्रो डेटा लोडरले पनि २ को सट्टा ३ मानहरू प्रदान गर्छ: पाठ र अफसेट भेक्टरहरू दुबै सुविधाका रूपमा प्रदान गरिन्छ। त्यसकारण, हामीले हाम्रो तालिम गर्ने कार्यलाई त्यस्तो समायोजन गर्न आवश्यक छ कि यसलाई हेर्न सकियोस्:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## सिमेन्टिक इम्बेडिङहरू: Word2Vec

हाम्रो अघिल्लो उदाहरणमा, मोडेलको इम्बेडिङ लेयरले शब्दहरूलाई भेक्टर प्रतिनिधित्वमा म्याप गर्न सिक्यो, यद्यपि, यस प्रतिनिधित्वमा धेरै सिमेन्टिक अर्थ थिएन। यस्तो भेक्टर प्रतिनिधित्व सिक्न राम्रो हुनेछ, जहाँ समान अर्थका शब्द वा पर्यायवाचीहरू भेक्टरहरूलाई कोहि नजिक हुने कुनै भेक्टर दूरी (जस्तै युक्लिडियन दूरी) अनुसार मेल खान्छन्।

त्यसका लागि, हामीले हाम्रो इम्बेडिङ मोडेललाई एउटा ठूलो पाठको सङ्कलनमा विशेष तरिकाले पूर्व-प्रशिक्षण गर्नुपर्छ। सिमेन्टिक इम्बेडिङहरू प्रशिक्षण गर्ने पहिलो तरिकाहरू मध्ये एउटा [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) हो। यो दुई मुख्य आर्किटेक्चरहरूमा आधारित छ जुन शब्दहरूको वितरण प्रतिनिधित्व उत्पादन गर्न प्रयोग गरिन्छ:

 - **कन्टिन्युअस ब्याग-आफ-वर्ड्स** (CBoW) — यस आर्किटेक्चरमा, हामी मोडेललाई वरिपरिका सन्दर्भबाट शब्द पूर्वानुमान गर्न प्रशिक्षित गर्छौं। दिइएको ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ मा, मोडेलको लक्ष्य $W_0$ लाई $(W_{-2},W_{-1},W_1,W_2)$ बाट पूर्वानुमान गर्नु हो।
 - **कन्टिन्युअस स्किप-ग्राम** CBoW को विपरित छ। मोडेलले वर्तमान शब्द पूर्वानुमान गर्न वरिपरिको सन्दर्भ शब्दहरूको विन्डो प्रयोग गर्छ।

CBoW छिटो हुन्छ, जबकि स्किप-ग्राम ढिलो हुन्छ, तर दुर्लभ शब्दहरूको प्रतिनिधित्व राम्रोसँग गर्छ।

![छवि जसले CBoW र स्किप-ग्राम एल्गोरिदमहरूलाई शब्दहरूलाई भेक्टरमा रूपान्तरण गर्न देखाउँछ।](../../../../../translated_images/ne/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

गुगल न्यूज डेटासेटमा पूर्व-प्रशिक्षित word2vec इम्बेडिङसंग प्रयोग गर्न, हामी **gensim** लाइब्रेरी प्रयोग गर्न सक्छौं। तल 'neural' सँग सबैभन्दा समान शब्दहरू फेला पारिएको छ।

> **टिप्पणी:** जब तपाईं पहिलो पटक शब्द भेक्टरहरू सिर्जना गर्नुहुन्छ, तिनीहरू डाउनलोड गर्न केही समय लाग्न सक्छ!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


हामी शब्दबाट भेक्टर एनबेडिङहरू पनि गणना गर्न सक्छौं, जुन वर्गीकरण मोडेल प्रशिक्षणमा प्रयोग गरिनेछ (स्पष्टताको लागि हामीले भेक्टरका पहिलो २० कम्पोनेन्ट मात्र देखाएका छौं):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

अर्थपूर्ण एम्बेडिङहरू बारे राम्रो कुरा के हो भने तपाईं भेक्टर इन्कोडिङलाई अर्थ परिवर्तन गर्नको लागि परिमार्जन गर्न सक्नुहुन्छ। उदाहरणका लागि, हामी सोध्न सक्छौं एउटा शब्द खोज्न, जसको भेक्टर प्रतिनिधित्व शब्दहरू *king* र *woman* लाई सकेसम्म नजिक र शब्द *man* बाट सकेसम्म टाढा हुनेछ:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

दुवै CBoW र Skip-Grams “पूर्वानुमानित” एम्बेडिङहरू हुन्, जसले केवल स्थानीय सन्दर्भहरूलाई मात्र ध्यानमा राख्दछन्। Word2Vec ले विश्वव्यापी सन्दर्भको फाइदा उठाउँदैन। 

**FastText**, Word2Vec मा आधारित छ जसले प्रत्येक शब्द र प्रत्येक शब्दमा पाइने क्याराक्टर n-grams का लागि भेक्टर प्रतिनिधित्वहरू सिक्छ। प्रतिनिधित्वहरूको मानहरू प्रत्येक प्रशिक्षण चरणमा एउटै भेक्टरमा औसत गरिन्छ। यो प्रि-ट्रेनिङमा धेरै थप गणना थप्छ भने यसले शब्द एम्बेडिङ्सलाई उप-शब्द जानकारी संहिताबद्ध गर्न सक्षम बनाउँछ। 

अर्को विधि, **GloVe**, सह-उत्पत्तिको म्याट्रिक्सको अवधारणालाई उपयोग गर्दछ, न्युरल विधिहरू प्रयोग गरी सह-उत्पत्तिको म्याट्रिक्सलाई अझ व्यक्तगामी र गैर-रैखिक शब्द भेक्टरहरूमा विच्छेदन गर्दछ।

तपाइँ उदाहरणसँग खेल्न सक्नुहुन्छ, embeddings लाई FastText र GloVe मा परिवर्तन गरेर, किनभने gensim ले विभिन्न शब्द एम्बेडिङ मोडेलहरूलाई समर्थन गर्दछ।


## PyTorch मा पूर्व-प्रशिक्षित एम्बेडिंगहरू प्रयोग गर्दै

हामी माथिको उदाहरणलाई परिमार्जन गरेर हाम्रो एम्बेडिंग तहमा पूर्व-प्रशिक्षित सेम्यान्टिकल एम्बेडिंगहरू जस्तै Word2Vec प्रयोग गर्न सक्छौं। हामीले ध्यान दिनुपर्छ कि पूर्व-प्रशिक्षित एम्बेडिंग र हाम्रो पाठ कोर्पसका शब्दावलीहरू सम्भवतः मेल खाँदैनन्, त्यसैले हामी हराएका शब्दहरूको तौलहरूलाई र्यान्डम मानहरूसँग आरम्भ गर्नेछौं:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


अब हाम्रो मोडेललाई तालिम दिनुहोस्। ध्यान दिनुहोस् कि मोडेललाई तालिम दिन लागेको समय अघिल्लो उदाहरणको तुलनामा उल्लेखनीय रूपमा ठूलो छ, किनभने ठूलो सम्प्रेषण तहको आकार र त्यसैले धेरै बढी प्यारामिटरहरू छन्। साथै, यसकारण, हामीलाई ओभरफिटिंगबाट बच्न चाहिने खण्डमा हाम्रो मोडेललाई बढी उदाहरणहरूमा तालिम दिनुपर्ने हुन सक्छ।


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

हाम्रो अवस्थामा हामी ठूलै वृद्धि देख्दैनौं शुद्धतामा, जुन सम्भवतः धेरै फरक शब्दावलीहरूको कारण हुनसक्छ। 
फरक शब्दावलीहरूको समस्या पार गर्न, हामी तलका मध्ये एउटा समाधान प्रयोग गर्न सक्छौं:
* हाम्रो शब्दावलीमा word2vec मोडेल पुनः तालिम दिनुहोस्
* पूर्व-तालिम गरिएको word2vec मोडेलबाट शब्दावली लोड गरेर हाम्रो डाटासेट लोड गर्नुहोस्। डाटासेट लोड गर्दा प्रयोग गरिएको शब्दावली निर्दिष्ट गर्न सकिन्छ।

पछिल्लो उपाय सहज देखिन्छ, विशेषगरी किनभने PyTorch `torchtext` फ्रेमवर्कमा एम्बेडिङका लागि पूर्वनिर्मित समर्थन छ। उदाहरणका लागि, हामी तलको तरिकाले GloVe-आधारित शब्दावली स्थापना गर्न सक्छौं:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

ती एम्बेडिङहरू प्रयोग गरेर वर्गीकरणकर्ता तालिम दिनको लागि, हामीले पहिले GloVe शब्दावली प्रयोग गरेर हाम्रो डेटासेटलाई इन्कोड गर्नुपर्छ:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

माथि हामीले हेरेका छौं कि सबै भेक्टर एम्बेडिङहरू `vocab.vectors` म्याट्रिक्समा संग्रहित छन्। यसले तिनीहरूलाई एमबेडिङ तहको तौलमा सरल प्रतिलिपि मार्फत लोड गर्न अत्यन्त सजिलो बनाउँछ:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

अब हाम्रो मोडेललाई तालिम दिने र हेर्ने कि के हामी राम्रो परिणाम पाउँछौं:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

हामी सटीकतामा उल्लेखनीय वृद्धी देखिरहेका छैनौं भन्नुको कारण मध्ये एउटा हो कि हाम्रो डेटासेटका केही शब्दहरू पूर्व-प्रशिक्षित GloVe शब्दसंग्रहमा छैनन्, र त्यसैले ती आधारभूत रूपमा बेवास्ता गरिन्छन्। यस तथ्यलाई पार गर्न हामी हाम्रो डेटासेटमा आधारित आफ्नै एम्बेडिङहरू प्रशिक्षण गर्न सक्छौं। 


## सन्दर्भात्मक एम्बेडिङहरू

परम्परागत पूर्व-प्रशिक्षित एम्बेडिङ प्रतिनिधित्वहरूको एउटा मुख्य सीमा जस्तै Word2Vec भनेको शब्द अर्थ अस्पष्टता समस्या हो। पूर्व-प्रशिक्षित एम्बेडिङहरूले सन्दर्भमा शब्दहरूको केहि अर्थ समात्न सक्छन्, तर शब्दको हरेक सम्भावित अर्थ एउटै एम्बेडिङमा संहिताबद्ध हुन्छ। यसले तलको मोडेलहरूमा समस्या ल्याउन सक्छ, किनभने धेरै शब्दहरू जस्तै 'play' शब्दको अर्थ उनीहरू प्रयोग गरिएको सन्दर्भमा निर्भर गर्दै फरक-फरक हुन्छ।

उदाहरणका लागि 'play' शब्द ती दुई फरक वाक्यहरूमा निकै फरक अर्थ राख्दछ:
- म थिएटरमा एक **play** हेर्न गएँ।
- जोन आफ्ना साथीहरूसँग **play** गर्न चाहन्छ।

माथि उल्लेखित पूर्व-प्रशिक्षित एम्बेडिङहरूले 'play' शब्दका यी दुवै अर्थहरू एउटै एम्बेडिङमा प्रतिनिधित्व गर्छन्। यस सीमालाई पार गर्नको लागि, हामीले **भाषा मोडेल** मा आधारित एम्बेडिङहरू बनाउन आवश्यक छ, जुन ठूलो पाठ संकलनमा प्रशिक्षण गरिएको हुन्छ, र *थाहा पाउँछ* कि शब्दहरू कसरी फरक सन्दर्भहरूमा जोडिन सक्छन्। सन्दर्भात्मक एम्बेडिङहरूको चर्चा यस ट्यूटोरियलको दायराभित्र छैन, तर भोलि भाषा मोडेलहरूको चर्चा गर्दा हामी तिनीहरूको पुन: समीक्षा गर्नेछौं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
